### Imports & Setup

In [1]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


In [2]:
# Import Step 11B outputs
%run ../11_machine_learning_attrition_prediction/03_train_test_split_and_scaling.ipynb

We reuse the same train-test split and preprocessing pipeline from step 11B to ensure fair model comparison.

### Model 1 – Decision Tree (Baseline Tree)

In [3]:
dt_model = DecisionTreeClassifier(
    max_depth=5,
    class_weight='balanced',
    random_state=42
)

dt_model.fit(X_train_scaled, y_train)

y_pred_dt = dt_model.predict(X_test_scaled)


### Evaluation

In [4]:
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))


Decision Tree Accuracy: 0.7721088435374149

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.82      0.86       247
           1       0.36      0.53      0.43        47

    accuracy                           0.77       294
   macro avg       0.63      0.67      0.64       294
weighted avg       0.81      0.77      0.79       294


Confusion Matrix:
 [[202  45]
 [ 22  25]]


#### Interpretation

The Decision Tree model captures non-linear relationships between features.
Compared to Logistic Regression, it often improves recall for attrition
but may overfit if not controlled.

### Model 2 – Random Forest (Main Model)

📌 Why Random Forest?

Ensemble of trees

Handles noise & imbalance better

Industry standard for HR analytics

In [5]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)


### Evaluation

In [6]:
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 0.8367346938775511

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.98      0.91       247
           1       0.44      0.09      0.14        47

    accuracy                           0.84       294
   macro avg       0.65      0.53      0.53       294
weighted avg       0.78      0.84      0.79       294


Confusion Matrix:
 [[242   5]
 [ 43   4]]


### Feature Importance

In [7]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train_scaled.columns
).sort_values(ascending=False)

feature_importance.head(10)


MonthlyIncome           0.076314
Age                     0.062581
TotalWorkingYears       0.053704
YearsAtCompany          0.052277
DailyRate               0.051400
OverTime_Yes            0.048873
MonthlyRate             0.046659
DistanceFromHome        0.044738
HourlyRate              0.043731
YearsWithCurrManager    0.041576
dtype: float64

Feature importance analysis reveals the key drivers of employee attrition.
These insights help HR teams focus on actionable factors rather than black-box predictions.


🌳 Tree-Based Models – Decision Tree & Random Forest Analysis
Objective

The objective of this step is to evaluate tree-based machine learning models for predicting employee attrition and to compare their performance against earlier baseline models.

Tree-based models are chosen because they:

Capture non-linear relationships

Handle feature interactions naturally

Provide feature importance for business interpretation

1️⃣ Decision Tree Classifier
Model Overview

A Decision Tree classifier was trained using the same preprocessed and scaled dataset used in the baseline models.

Performance Summary

Accuracy: ~77%

Attrition Recall (Yes): ~53%

Strengths:

Easy to interpret

Captures decision rules clearly

Limitations:

Still misses a significant number of attrition cases

Can overfit if not carefully tuned

Business Interpretation

The Decision Tree model demonstrates moderate ability to identify employees at risk of attrition.
While it improves interpretability compared to logistic regression, its recall performance suggests that additional optimization is required before practical HR deployment.

2️⃣ Random Forest Classifier
Model Overview

A Random Forest model was trained to improve predictive performance by combining multiple decision trees.

Performance Summary

Accuracy: ~84% (highest among all models)

Attrition Recall (Yes): ❌ Very low

Strengths:

Strong overall accuracy

Robust against overfitting

Limitations:

Severely biased toward predicting “No Attrition”

Poor detection of actual attrition cases due to class imbalance

Business Interpretation

Despite high accuracy, the Random Forest model fails to correctly identify employees who actually leave the organization.

From a business perspective, this model is not suitable in its current form because HR decision-making prioritizes identifying attrition risk—not simply predicting retention.

This highlights the critical importance of evaluating models beyond accuracy, especially in imbalanced classification problems.

3️⃣ Feature Importance Insights (Random Forest)

The Random Forest model provides valuable feature importance scores, revealing key drivers of employee attrition:

Top Influential Features

Monthly Income

Age

Total Working Years

Years at Company

Daily Rate

Overtime (Yes)

Distance from Home

Business Alignment

These features strongly align with earlier EDA and SQL analysis findings, confirming that:

Compensation

Work-life balance

Career stage

Commute burden

are primary contributors to employee attrition.

This consistency validates both the data preparation process and the learning behavior of the model.

4️⃣ Model Comparison Summary
Model	Accuracy	Attrition Recall	HR Suitability
Logistic Regression	Moderate	High	✅ Suitable
Decision Tree	Moderate	Medium	⚠️ Needs tuning
Random Forest	High	❌ Low	❌ Not suitable (current)
5️⃣ Key Takeaways

High accuracy alone does not indicate a good attrition model

Recall is more important than accuracy in HR risk prediction

Tree-based models offer strong interpretability and insight

Class imbalance must be addressed before final model selection

6️⃣ Next Step

Proceed to Model Tuning & Optimization, where:

Class imbalance will be addressed

Thresholds and parameters will be tuned

A business-optimized final model will be selected

In [8]:
import joblib

joblib.dump(dt_model, "decision_tree_model.pkl")
joblib.dump(rf_model, "random_forest_model.pkl")


['random_forest_model.pkl']